In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Loading the dataset and displaying the first 5 rows
file_path = r'/content/drive/MyDrive/project_source_files/netflix_titles.csv'
df=pd.read_csv(file_path)
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


# Data Profiling

The objective of this section is to evaluate the quality of the dataset before making any modifications.

The analysis focuses on:

- Missing values
- Duplicate records
- Data types
- Unique values
- Overall dataset quality

In [5]:
# Listing missing values in the dataset
missing_values=df.isnull().sum()
missing_values


,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [6]:
# Size of the dataset
data_size=df.shape
data_size

(8807, 12)

# Observation: 
The Dataset Contains 8,807 rows and 12 columns.

In [7]:
# Data Types and Summary Statistics
data_types=df.dtypes
data_info=df.info(verbose=True, show_counts=True)
print(data_types)
print(data_info)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object

In [8]:
# Finding Missing Values in the Dataset
missing_values=df.isnull().sum().sort_values(ascending=False)
missing_values

,0
director,2634
country,831
cast,825
date_added,10
rating,4
duration,3
show_id,0
type,0
title,0
release_year,0


## Missing Values Observation

### Columns with missing values are:

- Director: 2634 missing values
- Country: 831 missing values
- Cast: 825 missing values
- Date Added: 10 missing values
- rating: 4 missing 
- duration: 3 missing values

These columns will require further investigation before cleaning.

In [9]:
# Calculate the percentage of missing values in each column
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_percentages_sorted = missing_percentages.sort_values(ascending=False)
missing_percentages_sorted

,0
director,29.908028
country,9.435676
cast,9.367549
date_added,0.113546
rating,0.045418
duration,0.034064
show_id,0.000000
type,0.000000
title,0.000000
release_year,0.000000


## Missing Values Observation

### Columns with missing values are:
 
- Director: 2634 missing values
- Country: 831 missing values
- Cast: 825 missing values
- Date Added: 10 missing values
- rating: 4 missing 
- duration: 3 missing values
These columns will require further investigation before cleaning.

In [10]:
# Check for duplicate rows
print(f"The Dataset has {df.duplicated().sum()} duplicate rows.")

The Dataset has 0 duplicate rows.


In [11]:
df.nunique()

,0
show_id,8807
type,2
title,8807
director,4528
cast,7692
country,748
date_added,1767
release_year,74
rating,17
duration,220


In [12]:
# Count Some Columns Value using the Value_counts() method. This will give us an idea of the distribution of values in these columns. 
df["type"].value_counts()

,count
type,
Movie,6131
TV Show,2676


In [13]:
# Check the data type of the "date_added" column
df["date_added"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 8807 entries, 0 to 8806
Series name: date_added
Non-Null Count  Dtype 
--------------  ----- 
8797 non-null   object
dtypes: object(1)
memory usage: 68.9+ KB


In [14]:
# Make a copy of the original dataset to avoid modifying it directly
df_cleaned = df.copy()


In [15]:
# Standardize Column Names
df_cleaned.columns = df_cleaned.columns.str.strip().str.title().str.replace("_", " ")
df_cleaned.columns


Index(['Show Id', 'Type', 'Title', 'Director', 'Cast', 'Country', 'Date Added',
       'Release Year', 'Rating', 'Duration', 'Listed In', 'Description'],
      dtype='object')

### As we confirmed that this DataSet is Free of Duplicate Rows we are free to move forward to work with the rest


In [16]:
# Clean Text Columns 
text_columns=df_cleaned.select_dtypes(include=['object', "string"]).columns.tolist()
for ccolumn in text_columns:
    df_cleaned[ccolumn] = df_cleaned[ccolumn].str.strip().str.title()
df_cleaned.columns

Index(['Show Id', 'Type', 'Title', 'Director', 'Cast', 'Country', 'Date Added',
       'Release Year', 'Rating', 'Duration', 'Listed In', 'Description'],
      dtype='object')

In [17]:
# Dates Data Type check
df_cleaned["Date Added"].dtypes

dtype('O')

In [18]:
# Date Conversion: Convert the "Date Added" column to datetime format for easier analysis and manipulation.
df_cleaned["Date Added"]=pd.to_datetime(df_cleaned["Date Added"], errors='coerce')
df_cleaned["Date Added"].dtype

dtype('<M8[ns]')

In [19]:
# Handle Missing Values
df_cleaned.isnull().sum().sort_values(ascending=False)

,0
Director,2634
Country,831
Cast,825
Date Added,10
Rating,4
Duration,3
Show Id,0
Type,0
Title,0
Release Year,0


In [20]:
# Handle Missing Values
df_cleaned["Director"] = df_cleaned["Director"].fillna("Unknown")
df_cleaned["Country"] = df_cleaned["Country"].fillna("Unknown")
df_cleaned["Cast"] = df_cleaned["Cast"].fillna("Unknown")
df_cleaned["Date Added"] = df_cleaned["Date Added"].fillna("Unknown")
df_cleaned["Rating"] = df_cleaned["Rating"].fillna("Not Rated")
df_cleaned["Duration"] = df_cleaned["Duration"].fillna("Unknown")
df_cleaned.isnull().sum().sort_values(ascending=False)




,0
Show Id,0
Type,0
Title,0
Director,0
Cast,0
Country,0
Date Added,0
Release Year,0
Rating,0
Duration,0


In [29]:
# Validate the Cleaned Dataset
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Show Id       8807 non-null   object
 1   Type          8807 non-null   object
 2   Title         8807 non-null   object
 3   Director      8807 non-null   object
 4   Cast          8807 non-null   object
 5   Country       8807 non-null   object
 6   Date Added    8807 non-null   object
 7   Release Year  8807 non-null   int64 
 8   Rating        8807 non-null   object
 9   Duration      8807 non-null   object
 10  Listed In     8807 non-null   object
 11  Description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [ ]:
# Save the Clean Dataset
from google.colab import files
df_cleaned.to_csv("Netflix_titles_cleaned.csv", index=False)
files.download("Netflix_titles_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>